# Autoresearch-GLM Experiment Analysis

Analysis of autonomous feature-search results on the TaiwanCredit benchmark from `results.tsv`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Expected columns: commit, val_auc, status, description
df = pd.read_csv('results.tsv', sep='\t')
df['val_auc'] = pd.to_numeric(df['val_auc'], errors='coerce')
df['status'] = df['status'].astype(str).str.strip().str.upper()

if 'num_features' in df.columns:
    df['num_features'] = pd.to_numeric(df['num_features'], errors='coerce')

print(f'Total experiments: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)


In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')
print(f'Crashes: {n_crash}')


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    extra = ''
    if 'num_features' in row and pd.notna(row['num_features']):
        extra = f"  features={int(row['num_features'])}"
    print(f"  #{i:3d}  auc={row['val_auc']:.6f}{extra}  {row['description']}")


## Val AUC Over Time

Track how the best kept validation AUC evolves as experiments progress. Higher is better, so the frontier is the running maximum.


In [ ]:
def parse_description(desc):
    config = {}
    for token in str(desc).split():
        if '=' not in token:
            continue
        key, value = token.split('=', 1)
        config[key] = value
    return {
        'screen_k': int(config.get('screen_k', 0)),
        'feature_cap': int(config.get('feature_cap', 0)),
        'interaction_cap': int(config.get('interaction_cap', 0)),
        'clip': config.get('clip', 'none'),
        'l2': config.get('l2', '0.000'),
        'transforms': config.get('transforms', 'identity').split('+'),
    }

def short_label(current, previous, is_last=False):
    if previous is None:
        return 'baseline'

    parts = []
    if current['clip'] != previous['clip'] and current['clip'] != 'none':
        parts.append(current['clip'])

    new_terms = [
        term for term in current['transforms']
        if term != 'identity' and term not in previous['transforms']
    ]
    if new_terms:
        parts.append('+' + '+'.join(new_terms))

    if current['interaction_cap'] != previous['interaction_cap']:
        parts.append(f"+{current['interaction_cap']}i")
    if current['screen_k'] != previous['screen_k']:
        parts.append(f"s{current['screen_k']}")
    if current['feature_cap'] != previous['feature_cap']:
        parts.append(f"cap{current['feature_cap']}")
    if current['l2'] != previous['l2']:
        short_l2 = current['l2'].replace('0.', '.')
        parts.append(f"l2 {short_l2}")

    if not parts:
        return 'best' if is_last else 'tune'
    return ', '.join(parts[:2])

fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_auc = valid.loc[0, 'val_auc']
discarded = valid[valid['status'] == 'DISCARD']
kept = valid[valid['status'] == 'KEEP'].copy()
kept['running_best'] = kept['val_auc'].cummax()
kept['delta'] = kept['running_best'].diff().fillna(0.0)

labels = []
previous = None
kept_configs = []
for _, row in kept.iterrows():
    config = parse_description(row['description'])
    kept_configs.append(config)
for i, config in enumerate(kept_configs):
    previous = kept_configs[i - 1] if i > 0 else None
    labels.append(short_label(config, previous, is_last=i == len(kept_configs) - 1))
kept['label'] = labels

milestone_idx = {kept.index[0], kept.index[-1]}
extra = kept.iloc[1:-1].nlargest(min(2, max(len(kept) - 2, 0)), 'delta')
milestone_idx.update(extra.index.tolist())
milestones = kept.loc[sorted(milestone_idx)]

ax.scatter(discarded.index, discarded['val_auc'], c='#cccccc', s=14, alpha=0.45, zorder=2, label='Discarded')
ax.scatter(kept.index, kept['val_auc'], c='#2ecc71', s=52, zorder=4, label='Kept', edgecolors='black', linewidths=0.5)
ax.step(kept.index, kept['running_best'], where='post', color='#27ae60', linewidth=2.2, alpha=0.8, zorder=3, label='Running best')
ax.axhline(baseline_auc, color='#34495e', linestyle='--', linewidth=1.5, alpha=0.8, label=f'Baseline ({baseline_auc:.4f})')

for n, (milestone_idx, row) in enumerate(milestones.iterrows()):
    is_first = milestone_idx == kept.index[0]
    is_last = milestone_idx == kept.index[-1]
    if is_last:
        x_offset, y_offset, align = -10, 8, 'right'
    elif is_first:
        x_offset, y_offset, align = 4, 6, 'left'
    else:
        x_offset, y_offset, align = 4, 10 + 5 * (n % 2), 'left'
    ax.annotate(
        row['label'],
        (milestone_idx, row['val_auc']),
        xytext=(x_offset, y_offset),
        textcoords='offset points',
        ha=align,
        fontsize=8.5,
        rotation=24,
        color='#2e8b57',
        bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7),
    )

ax.set_title(f'Autoresearch-GLM Progress: {len(valid)} Experiments, {len(kept)} Kept Improvements')
ax.set_xlabel('Experiment index')
ax.set_ylabel('Validation AUC')
ax.grid(True, alpha=0.2)
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()


## Summary Statistics


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline_auc = df.iloc[0]['val_auc']
best_auc = kept['val_auc'].max()
best_row = kept.loc[kept['val_auc'].idxmax()]

print(f'Baseline val_auc:  {baseline_auc:.6f}')
print(f'Best val_auc:      {best_auc:.6f}')
print(f'Total improvement: {best_auc - baseline_auc:+.6f} ({(best_auc - baseline_auc) / baseline_auc * 100:.2f}%)')
print(f"Best experiment:   {best_row['description']}")
print()
print('Cumulative frontier improvements:')
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    print(f"  Experiment #{int(row['index']):3d}: auc={row['val_auc']:.6f}  {row['description']}")


## Top Hits (Kept Experiments by Improvement)


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
kept['prev_auc'] = kept['val_auc'].shift(1)
kept['delta'] = kept['val_auc'] - kept['prev_auc']
hits = kept.iloc[1:].copy().sort_values('delta', ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'AUC':>10}  Description")
print('-' * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_auc']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")
